In [1]:
import pandas as pd
import numpy as np

# 1. Đọc file dữ liệu gốc bạn vừa tải lên
data = pd.read_csv('D:\\VisualStudio\\IE212\\DoAn\Air Quality Ho Chi Minh City.csv')
data

,date,Station_No,TSP,PM2.5,O3,CO,NO2,SO2,Temperature,Humidity
0,23-02-2021 21:00,1,32.935714,15.604762,55.431381,1330.451429,112.740762,393.000000,28.361905,63.188095
1,23-02-2021 22:00,1,30.932353,14.594118,58.197176,1200.603529,112.366471,377.588235,28.320588,63.773529
2,23-02-2021 23:00,1,27.645000,13.436667,55.029433,1177.897000,112.700433,372.476667,28.336667,64.205000
3,24-02-2021 00:00,1,24.380000,12.365000,54.767700,1267.476000,112.480867,389.070000,28.305000,64.735000
4,24-02-2021 01:00,1,22.521667,11.636667,53.786200,1322.293000,114.331500,393.000000,28.300000,65.188333
...,...,...,...,...,...,...,...,...,...,...
52543,21-06-2022 13:00,6,21.526667,10.201667,100.080283,1007.907000,73.962600,146.720000,33.496667,60.470000
52544,21-06-2022 14:00,6,26.590000,11.250000,119.612133,1262.319000,96.609333,181.216667,33.026667,62.556667
52545,21-06-2022 15:00,6,27.606667,11.355000,119.448550,1457.330000,100.310600,212.220000,33.106667,62.175000
52546,21-06-2022 16:00,6,37.433333,15.048333,125.730150,2125.257000,128.383767,350.643333,31.406667,70.605000


In [ ]:
data['datetime'] = pd.to_datetime(data['date'], format='%d-%m-%Y %H:%M')


In [3]:
data["Station_No"].value_counts(dropna=False)

Station_No
4    9951
6    9499
2    9357
3    8418
1    7892
5    7431
Name: count, dtype: int64

In [4]:
data.groupby('Station_No').apply(lambda x: x.isnull().sum())

,date,Station_No,TSP,PM2.5,O3,CO,NO2,SO2,Temperature,Humidity,datetime
Station_No,,,,,,,,,,,
1,0,0,60,0,5665,65,5665,5709,0,0,0
2,0,0,0,0,0,8999,0,9,0,0,0
3,0,0,0,0,1,1,0,28,0,0,0
4,0,0,0,0,36,0,0,12,0,0,0
5,0,0,0,0,4908,0,1,4945,4437,4432,0
6,0,0,0,0,0,0,0,303,0,0,0


In [5]:
data['date'] = pd.to_datetime(data['date'], format='%d-%m-%Y %H:%M')
cutoff = {
    1: pd.Timestamp('2021-10-09 15:00:00'),  # 1 giờ trước block null bắt đầu
    5: pd.Timestamp('2021-10-11 14:00:00'),
}
 
dfs = []
for station in sorted(data['Station_No'].unique()):
    sub = data[data['Station_No'] == station].copy()
 
    if station in cutoff:
        before = len(sub)
        sub = sub[sub['date'] <= cutoff[station]]
        after = len(sub)
        print(f"  Trạm {station}: cắt tại {cutoff[station]} "
              f"({before} → {after} bản ghi, bỏ {before-after})")
 
    # Trạm 2: loại cột CO (96% null)
    if station == 2:
        sub['CO'] = np.nan
        print(f"  Trạm 2: đặt CO = NaN toàn bộ (cảm biến không hoạt động)")
 
    dfs.append(sub)

data = pd.concat(dfs, ignore_index=True)
data = data.sort_values(['Station_No', 'date']).reset_index(drop=True)
data["Station_No"].value_counts(dropna=False)

  Trạm 1: cắt tại 2021-10-09 15:00:00 (7892 → 2227 bản ghi, bỏ 5665)
  Trạm 2: đặt CO = NaN toàn bộ (cảm biến không hoạt động)
  Trạm 5: cắt tại 2021-10-11 14:00:00 (7431 → 2523 bản ghi, bỏ 4908)


Station_No
4    9951
6    9499
2    9357
3    8418
5    2523
1    2227
Name: count, dtype: int64

In [ ]:
data['date'] = pd.to_datetime(data['date'], format='%d-%m-%Y %H:%M')
stream_data = data[
    (data['date'] >= '2021-02-24 00:00') &
    (data['date'] <= '2021-07-27 00:00')
]

stream_data.groupby('Station_No').apply(lambda x: x.isnull().sum())

,date,Station_No,TSP,PM2.5,O3,CO,NO2,SO2,Temperature,Humidity,datetime
Station_No,,,,,,,,,,,
1,0,0,0,0,0,0,0,2,0,0,0
2,0,0,0,0,0,2737,0,9,0,0,0
3,0,0,0,0,0,1,0,20,0,0,0
4,0,0,0,0,8,0,0,3,0,0,0
5,0,0,0,0,0,0,0,38,0,0,0
6,0,0,0,0,0,0,0,204,0,0,0


In [ ]:
stream_data["Station_No"].value_counts(dropna=False)

Station_No
4    3056
2    2737
6    2723
3    2648
5    2506
1    2179
Name: count, dtype: int64

In [ ]:
import numpy as np
##BƯỚC 1: LOẠI GIÁ TRỊ PHI THỰC
valid_range = {
    'PM2.5':       (0, 500),      
    'TSP':         (0, 1000),     
    'O3':          (0, 500),      
    'CO':          (0, 15000),    
    'NO2':         (0, 500),      
    'SO2':         (0, 700),      
    'Temperature': (-10, 50),     
    'Humidity':    (0, 100),      
}

total_outliers = 0
for station in sorted(stream_data['Station_No'].unique()):
    idx = stream_data[stream_data['Station_No'] == station].index
    pm = stream_data.loc[idx, 'PM2.5'].copy()
    pm_prev = pm.shift(1)
 
    outlier_mask = (pm > 200) & (pm > 3 * pm_prev)
    n = outlier_mask.sum()
    total_outliers += n
 
    stream_data.loc[idx[outlier_mask], 'PM2.5'] = np.nan
    if n > 0:
        print(f"  Trạm {station}: {n} outlier bị loại")

  Trạm 1: 1 outlier bị loại
  Trạm 6: 1 outlier bị loại


In [ ]:
FILL_FEATURES = ['PM2.5', 'TSP', 'O3', 'NO2', 'SO2', 'Temperature', 'Humidity']
MAX_GAP_FILL = 24  # giờ — không fill gap dài hơn 24h
 
def fill_short_gaps(series, max_gap=24):
    """
    Fill null theo thứ tự:
    1. Gap = 1h: dùng giá trị liền trước (forward fill 1 bước)
    2. Gap > 1h đến max_gap: dùng cùng giờ ngày hôm trước (lag 24h)
    Không fill gap dài hơn max_gap.
    """
    s = series.copy()
    
    # Đánh dấu vị trí null ban đầu
    null_mask = s.isnull()
    
    if not null_mask.any():
        return s, 0, 0
    
    # Tính độ dài gap tại mỗi vị trí null
    # Gán group id cho mỗi khối null liên tục
    group = (null_mask != null_mask.shift()).cumsum()
    gap_lengths = null_mask.groupby(group).transform('sum')
    
    filled_ffill = 0
    filled_lag24 = 0
    
    # Gap = 1h → forward fill
    mask_1h = null_mask & (gap_lengths == 1)
    s[mask_1h] = s.shift(1)[mask_1h]
    filled_ffill = mask_1h.sum()
    
    # Cập nhật null_mask
    null_mask = s.isnull()
    group = (null_mask != null_mask.shift()).cumsum()
    gap_lengths = null_mask.groupby(group).transform('sum')
    
    # Gap 2h đến max_gap → dùng lag 24h
    mask_lag = null_mask & (gap_lengths > 1) & (gap_lengths <= max_gap)
    lag24 = s.shift(24)
    s[mask_lag] = lag24[mask_lag]
    filled_lag24 = mask_lag.sum()
 
    return s, filled_ffill, filled_lag24
 
null_before_fill = stream_data[FILL_FEATURES].isnull().sum().sum()
 
results = []
for station in sorted(stream_data['Station_No'].unique()):
    idx = stream_data[stream_data['Station_No'] == station].index
    sub = stream_data.loc[idx].copy()
    
    station_ffill = 0
    station_lag24 = 0
    station_unfilled = 0
    
    for col in FILL_FEATURES:
        # Bỏ qua CO trạm 2
        if station == 2 and col == 'CO':
            continue
            
        filled, n_ffill, n_lag24 = fill_short_gaps(sub[col], MAX_GAP_FILL)
        sub[col] = filled
        station_ffill += n_ffill
        station_lag24 += n_lag24
    
    station_unfilled = sub[FILL_FEATURES].isnull().sum().sum()
    stream_data.loc[idx, FILL_FEATURES] = sub[FILL_FEATURES]
    
    print(f"  Trạm {station}: fill ffill={station_ffill}, "
          f"lag24={station_lag24}, còn null={station_unfilled}")

  Trạm 1: fill ffill=3, lag24=0, còn null=0
  Trạm 2: fill ffill=7, lag24=2, còn null=0
  Trạm 3: fill ffill=8, lag24=12, còn null=2
  Trạm 4: fill ffill=3, lag24=8, còn null=0
  Trạm 5: fill ffill=24, lag24=14, còn null=0
  Trạm 6: fill ffill=42, lag24=163, còn null=40


In [32]:
import pandas as pd

# 1. Đọc file dữ liệu gốc
df = pd.read_csv('D:\VisualStudio\IE212\DoAn\Air Quality Ho Chi Minh City.csv')

# 2. Chuyển cột date sang dạng datetime để lọc chính xác
df['parsed_date'] = pd.to_datetime(df['date'], format='%d-%m-%Y %H:%M')

# 3. Giới hạn khoảng thời gian bạn yêu cầu (24/02/2021 đến 27/07/2021)
start_date = pd.to_datetime('2021-02-24 00:00:00')
end_date = pd.to_datetime('2021-07-27 23:59:59')
df_filtered = df[(df['parsed_date'] >= start_date) & (df['parsed_date'] <= end_date)]

# 4. Tìm các mốc giờ mà CẢ 6 TRẠM ĐỀU XUẤT HIỆN ĐẦY ĐỦ
date_counts = df_filtered.groupby('parsed_date')['Station_No'].nunique()
valid_dates = date_counts[date_counts == 6].index

# 5. Lọc và xếp dữ liệu theo trình tự thời gian tăng dần
stream_data = df_filtered[df_filtered['parsed_date'].isin(valid_dates)]
stream_data = stream_data.sort_values(by=['parsed_date', 'Station_No'])

# 6. Kiểm tra lại số lượng mẫu sau khi đồng bộ
print("--- SỐ LƯỢNG MẪU MỖI TRẠM SAU KHI ĐỒNG BỘ ---")
print(stream_data['Station_No'].value_counts())

# 7. Xuất ra file simulation_stream.csv phục vụ cho Producer bắn Kafka
stream_data = stream_data.drop(columns=['parsed_date'])


print("\n[+] Đã làm xong file data/simulation_stream.csv đồng bộ hoàn hảo!")

--- SỐ LƯỢNG MẪU MỖI TRẠM SAU KHI ĐỒNG BỘ ---
Station_No
1    1081
2    1081
3    1081
4    1081
5    1081
6    1081
Name: count, dtype: int64

[+] Đã làm xong file data/simulation_stream.csv đồng bộ hoàn hảo!


In [34]:
stream_data.groupby('Station_No').apply(lambda x: x.isnull().sum())

,date,Station_No,TSP,PM2.5,O3,CO,NO2,SO2,Temperature,Humidity
Station_No,,,,,,,,,,
1,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,1069,0,4,0,0
3,0,0,0,0,0,0,0,3,0,0
4,0,0,0,0,8,0,0,0,0,0
5,0,0,0,0,0,0,0,19,0,0
6,0,0,0,0,0,0,0,79,0,0


In [35]:
import numpy as np
##BƯỚC 1: LOẠI GIÁ TRỊ PHI THỰC
valid_range = {
    'PM2.5':       (0, 500),      
    'TSP':         (0, 1000),     
    'O3':          (0, 500),      
    'CO':          (0, 15000),    
    'NO2':         (0, 500),      
    'SO2':         (0, 700),      
    'Temperature': (-10, 50),     
    'Humidity':    (0, 100),      
}

total_outliers = 0
for station in sorted(stream_data['Station_No'].unique()):
    idx = stream_data[stream_data['Station_No'] == station].index
    pm = stream_data.loc[idx, 'PM2.5'].copy()
    pm_prev = pm.shift(1)
 
    outlier_mask = (pm > 200) & (pm > 3 * pm_prev)
    n = outlier_mask.sum()
    total_outliers += n
 
    stream_data.loc[idx[outlier_mask], 'PM2.5'] = np.nan
    if n > 0:
        print(f"  Trạm {station}: {n} outlier bị loại")

In [42]:
FILL_FEATURES = ['PM2.5', 'TSP', 'O3', 'NO2', 'SO2', 'Temperature', 'Humidity']
MAX_GAP_FILL = 24  # giờ — không fill gap dài hơn 24h
 
def fill_short_gaps(series, max_gap=24):
    """
    Fill null theo thứ tự:
    1. Gap = 1h: dùng giá trị liền trước (forward fill 1 bước)
    2. Gap > 1h đến max_gap: dùng cùng giờ ngày hôm trước (lag 24h)
    Không fill gap dài hơn max_gap.
    """
    s = series.copy()
    
    # Đánh dấu vị trí null ban đầu
    null_mask = s.isnull()
    
    if not null_mask.any():
        return s, 0, 0
    
    # Tính độ dài gap tại mỗi vị trí null
    # Gán group id cho mỗi khối null liên tục
    group = (null_mask != null_mask.shift()).cumsum()
    gap_lengths = null_mask.groupby(group).transform('sum')
    
    filled_ffill = 0
    filled_lag24 = 0
    
    # Gap = 1h → forward fill
    mask_1h = null_mask & (gap_lengths == 1)
    s[mask_1h] = s.shift(1)[mask_1h]
    filled_ffill = mask_1h.sum()
    
    # Cập nhật null_mask
    null_mask = s.isnull()
    group = (null_mask != null_mask.shift()).cumsum()
    gap_lengths = null_mask.groupby(group).transform('sum')
    
    # Gap 2h đến max_gap → dùng lag 24h
    mask_lag = null_mask & (gap_lengths > 1) & (gap_lengths <= max_gap)
    lag24 = s.shift(24)
    s[mask_lag] = lag24[mask_lag]
    filled_lag24 = mask_lag.sum()
 
    return s, filled_ffill, filled_lag24
 
null_before_fill = stream_data[FILL_FEATURES].isnull().sum().sum()
 
results = []
for station in sorted(stream_data['Station_No'].unique()):
    idx = stream_data[stream_data['Station_No'] == station].index
    sub = stream_data.loc[idx].copy()
    
    station_ffill = 0
    station_lag24 = 0
    station_unfilled = 0
    
    for col in FILL_FEATURES:
        # Bỏ qua CO trạm 2
        if station == 2 and col == 'CO':
            continue
            
        filled, n_ffill, n_lag24 = fill_short_gaps(sub[col], MAX_GAP_FILL)
        sub[col] = filled
        station_ffill += n_ffill
        station_lag24 += n_lag24
    
    station_unfilled = sub[FILL_FEATURES].isnull().sum().sum()
    stream_data.loc[idx, FILL_FEATURES] = sub[FILL_FEATURES]
    
    print(f"  Trạm {station}: fill ffill={station_ffill}, "
          f"lag24={station_lag24}, còn null={station_unfilled}")

  Trạm 1: fill ffill=0, lag24=0, còn null=0
  Trạm 2: fill ffill=0, lag24=0, còn null=0
  Trạm 3: fill ffill=0, lag24=0, còn null=0
  Trạm 4: fill ffill=0, lag24=0, còn null=0
  Trạm 5: fill ffill=0, lag24=0, còn null=0
  Trạm 6: fill ffill=0, lag24=3, còn null=3


In [45]:
stream_data['SO2'] = stream_data.groupby('Station_No')['SO2'].ffill()
# Nếu vẫn còn sót (do null ở ngay dòng đầu tiên), điền bằng 0 hoặc median
stream_data['SO2'] = stream_data['SO2'].fillna(0.0)

# 7. XỬ LÝ CỘT CO CHO STATION 2: Điền bằng 0.0 vì không dùng làm feature dự báo
stream_data.loc[stream_data['Station_No'] == 2, 'CO'] = 0.0
stream_data.groupby('Station_No').apply(lambda x: x.isnull().sum())

,date,Station_No,TSP,PM2.5,O3,CO,NO2,SO2,Temperature,Humidity
Station_No,,,,,,,,,,
1,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0
5,0,0,0,0,0,0,0,0,0,0
6,0,0,0,0,0,0,0,0,0,0


In [48]:
stream_data.to_csv('D:\VisualStudio\IE212\DoAn\streaming\data\simulation_stream.csv', index=False)